# 連続時間拡散モデルとフローマッチング

連続時間生成では、単純なノイズ分布からデータ分布へ点を少しずつ運ぶ流れを学びます。ここで重要なのは、完成画像や最終サンプルだけを真似るのではなく、途中の時刻ごとに「今いる場所からどちらへ動けばデータらしくなるか」を学ぶ点です。

Flow Matching は、途中点 `x_t` と目標速度 `u_t` を作り、`v_theta(x_t, t)` が `u_t` に近づくように回帰します。拡散モデルはノイズ除去やスコアを通じて動き方を得ますが、Flow Matching は経路と速度を直接設計し、ODE `dx/dt = v_theta(x,t)` を積分して生成します。この教材では、path の作り方、時刻条件、速度場の誤差、積分した後の分布のずれを分けて確認します。

In [ ]:
import math
import random
import statistics

random.seed(41)


def sample_noise(n):
    return [(random.gauss(0.0, 1.0), random.gauss(0.0, 1.0)) for _ in range(n)]


def sample_data(n, rng=random):
    out = []
    for _ in range(n):
        if rng.random() < 0.52:
            out.append((rng.gauss(-2.0, 0.38), rng.gauss(0.8, 0.35)))
        else:
            out.append((rng.gauss(1.7, 0.45), rng.gauss(-0.9, 0.42)))
    return out

noise_ref = sample_noise(1600)
data_ref = sample_data(1600)


def describe(points):
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    return {
        'mx': round(statistics.mean(xs), 3),
        'my': round(statistics.mean(ys), 3),
        'sx': round(statistics.pstdev(xs), 3),
        'sy': round(statistics.pstdev(ys), 3),
        'left': round(sum(x < 0 for x in xs) / len(xs), 3),
    }

print('noise:', describe(noise_ref))
print('data:', describe(data_ref))

線形 path は `x_t = (1-t)x0 + t x1` で、目標速度は一定の `u_t = x1 - x0` になる。三角 path は係数を `sin` と `cos` に変え、序盤と終盤の混ざり方を変える。path を変えると、同じノイズ点とデータ点を結んでいても途中点と速度の分布が変わる。

In [ ]:
def linear_path(x0, x1, t):
    xt = ((1.0 - t) * x0[0] + t * x1[0], (1.0 - t) * x0[1] + t * x1[1])
    ut = (x1[0] - x0[0], x1[1] - x0[1])
    return xt, ut


def trig_path(x0, x1, t):
    alpha = math.sin(0.5 * math.pi * t)
    sigma = math.cos(0.5 * math.pi * t)
    dalpha = 0.5 * math.pi * math.cos(0.5 * math.pi * t)
    dsigma = -0.5 * math.pi * math.sin(0.5 * math.pi * t)
    xt = (sigma * x0[0] + alpha * x1[0], sigma * x0[1] + alpha * x1[1])
    ut = (dsigma * x0[0] + dalpha * x1[0], dsigma * x0[1] + dalpha * x1[1])
    return xt, ut

x0 = noise_ref[0]
x1 = data_ref[0]
for t in [0.0, 0.25, 0.5, 0.75, 1.0]:
    xl, ul = linear_path(x0, x1, t)
    xt, ut = trig_path(x0, x1, t)
    print(t, 'linear x/u', tuple(round(v, 2) for v in xl), tuple(round(v, 2) for v in ul))
    print(t, 'trig   x/u', tuple(round(v, 2) for v in xt), tuple(round(v, 2) for v in ut))

訓練データは `(x_t, t) -> u_t` の回帰問題になる。最終サンプルを直接比べるのではなく、途中の各時刻で進むべき方向を学ぶ。つまりモデルは『この場所と時刻なら、次にどちらへ進むべきか』を近似している。

In [ ]:
def make_pairs(n, path_fn, seed=100):
    rng = random.Random(seed)
    rows = []
    for _ in range(n):
        x0 = (rng.gauss(0.0, 1.0), rng.gauss(0.0, 1.0))
        x1 = sample_data(1, rng)[0]
        t = rng.random()
        xt, ut = path_fn(x0, x1, t)
        rows.append((xt[0], xt[1], t, ut[0], ut[1]))
    return rows

train_rows = make_pairs(2400, linear_path)
print('row:', tuple(round(v, 3) for v in train_rows[0]))
print('target speed mean:', round(statistics.mean(r[3] for r in train_rows), 3), round(statistics.mean(r[4] for r in train_rows), 3))

速度場は小さな線形回帰器で表す。入力特徴には位置、時刻、位置と時刻の相互作用を入れる。`t` が入るため、同じ位置でも時刻によって別の速度を出せる。

In [ ]:
def features(x, y, t, use_time=True):
    if use_time:
        return [1.0, x, y, t, x * t, y * t, t * t, x * x, y * y]
    return [1.0, x, y, x * x, y * y, x * y, abs(x), abs(y), x + y]


def predict(theta, x, y, t, use_time=True):
    phi = features(x, y, t, use_time)
    vx = sum(w * p for w, p in zip(theta[0], phi))
    vy = sum(w * p for w, p in zip(theta[1], phi))
    return vx, vy


def train_field(rows, use_time=True, steps=420, lr=0.018):
    dim = len(features(0.0, 0.0, 0.0, use_time))
    theta = [[random.gauss(0.0, 0.03) for _ in range(dim)] for _ in range(2)]
    history = []
    for step in range(steps):
        grad = [[0.0 for _ in range(dim)] for _ in range(2)]
        loss = 0.0
        batch = random.sample(rows, 96)
        for x, y, t, ux, uy in batch:
            phi = features(x, y, t, use_time)
            vx, vy = predict(theta, x, y, t, use_time)
            ex = vx - ux
            ey = vy - uy
            loss += ex * ex + ey * ey
            for j, p in enumerate(phi):
                grad[0][j] += 2.0 * ex * p / len(batch)
                grad[1][j] += 2.0 * ey * p / len(batch)
        for axis in range(2):
            for j in range(dim):
                theta[axis][j] -= lr * grad[axis][j]
        if step % 70 == 0 or step == steps - 1:
            history.append((step, loss / len(batch)))
    return theta, history

lin_theta, lin_history = train_field(train_rows, use_time=True)
print([(s, round(v, 3)) for s, v in lin_history])

生成では、ノイズ点から始めて `dx/dt = v_theta(x,t)` を Euler 法で積分する。訓練時の教師速度を使わず、学習した速度場だけで点を移動させる。訓練は局所的な速度の回帰だが、生成ではその小さな移動を何度も積み重ねて分布全体を動かす。

In [ ]:
def integrate(theta, n=1600, ode_steps=80, use_time=True, seed=700):
    rng = random.Random(seed)
    points = [(rng.gauss(0.0, 1.0), rng.gauss(0.0, 1.0)) for _ in range(n)]
    dt = 1.0 / ode_steps
    for k in range(ode_steps):
        t = k / ode_steps
        moved = []
        for x, y in points:
            vx, vy = predict(theta, x, y, t, use_time)
            moved.append((max(-5.0, min(5.0, x + dt * vx)), max(-5.0, min(5.0, y + dt * vy))))
        points = moved
    return points

lin_samples = integrate(lin_theta)

print('generated:', describe(lin_samples))
print('target:', describe(data_ref))

path を変えると教師速度の分布も変わる。同じ回帰器でも、線形 path と三角 path では学ぶべき速度場が異なる。

In [ ]:
trig_rows = make_pairs(2400, trig_path, seed=101)
trig_theta, trig_history = train_field(trig_rows, use_time=True)
trig_samples = integrate(trig_theta, seed=701)

print('linear history:', [(s, round(v, 3)) for s, v in lin_history[-3:]])
print('trig history:', [(s, round(v, 3)) for s, v in trig_history[-3:]])
print('linear samples:', describe(lin_samples))
print('trig samples:', describe(trig_samples))

時刻入力を消すと、同じ位置に対して全時刻で同じ速度しか返せない。連続時間生成では序盤と終盤で必要な動きが変わるため、この制約は性能を落としやすい。時刻は単なる付録ではなく、ノイズをほどく段階を表す条件である。

In [ ]:
no_t_theta, no_t_history = train_field(train_rows, use_time=False)
no_t_samples = integrate(no_t_theta, use_time=False, seed=702)

print('with t:', describe(lin_samples))
print('no t:', describe(no_t_samples))
print('with t loss:', round(lin_history[-1][1], 3))
print('no t loss:', round(no_t_history[-1][1], 3))

OT-CFM は回帰損失の形より、ノイズ点とデータ点の対応を変える。遠い相手と無理に結ぶと速度が大きく乱れやすい。近い相手へ貪欲に割り当てるだけでも、教師速度の長さは小さくなる。

In [ ]:
def distance(a, b):
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)


def greedy_pair(noise, data):
    unused = data[:]
    pairs = []
    for x0 in noise:
        best_i = min(range(len(unused)), key=lambda i: distance(x0, unused[i]))
        pairs.append((x0, unused.pop(best_i)))
    return pairs

small_noise = sample_noise(90)
small_data = sample_data(90)
random_pairs = list(zip(small_noise, small_data))
greedy_pairs = greedy_pair(small_noise, small_data)

for name, pairs in [('random', random_pairs), ('greedy', greedy_pairs)]:
    lengths = [distance(a, b) for a, b in pairs]
    print(name, 'mean length:', round(statistics.mean(lengths), 3), 'max:', round(max(lengths), 3))

Flow Matching は、途中点での速度回帰として生成を作る。path は教師速度を決め、時刻入力は速度場の段階差を表し、ODE 積分がノイズ点をデータ側へ運ぶ。拡散モデルと同じく連続時間の点の移動を扱うが、速度を直接教師あり回帰で学ぶ点が特徴になる。